# NHL Analytics Hub - Part 2: Fetching from the API and loading MySQL

This notebook calls the NHL API in a loop and inserts each response straight into the MySQL database `nhl_db` with PyMySQL. No files are written in between. Every response is flattened into a pandas DataFrame, cleaned (type conversions, duplicates removed, foreign keys checked) and then inserted.

Load order (parents before children):
`teams` -> `standings` -> `players` -> `games` -> `skater_season_stats` / `goalie_season_stats` -> `game_stats`

Rows are inserted with `INSERT IGNORE`, so anything that is already in the table (same primary or unique key) is skipped. That handles the duplicates the API naturally returns (every game is in both teams' schedules) and makes the notebook safe to re-run.

Database settings can be overridden with the environment variables `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_PASSWORD` and `DB_NAME`. The defaults match a local WAMP install where MySQL 8 runs on port **3308** (MariaDB sits on 3306).

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import pymysql
import requests

BASE_URL = "https://api-web.nhle.com/v1"
SEASON = "20252026"
DELAY = 0.5  # seconds to wait before each request
SQL_FILE = Path("../sql/create_tables.sql")

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("DB_PORT", "3308")),
    "user": os.getenv("DB_USER", "root"),
    "password": os.getenv("DB_PASSWORD", ""),
    "database": os.getenv("DB_NAME", "nhl_db"),
}

## Helper functions

In [2]:
def get_json(url, retries=3):
    for attempt in range(1, retries + 1):
        time.sleep(DELAY)
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as err:
            print(f"Attempt {attempt} failed for {url}: {err}")
            time.sleep(attempt * 2)
    return None


def name(value):
    # names in the API look like {"default": "Nathan", "cs": ...}
    if isinstance(value, dict):
        return value.get("default")
    return value


def toi_to_mmss(seconds):
    if seconds is None or pd.isna(seconds):
        return None
    total = int(round(seconds))
    return f"{total // 60:02d}:{total % 60:02d}"


def make_frame(rows, key, ints=(), floats=(), dates=()):
    """Flat rows -> DataFrame with proper types and no duplicate keys."""
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for col in ints:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in floats:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in dates:
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.date
    return df.drop_duplicates(subset=key).reset_index(drop=True)


def insert_rows(cursor, table, df):
    if df.empty:
        return 0
    columns = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT IGNORE INTO {table} ({columns}) VALUES ({placeholders})"
    # MySQL needs None, not NaN / pd.NA, for NULL values
    rows = df.astype(object).where(df.notna(), None).values.tolist()
    cursor.executemany(sql, rows)
    return cursor.rowcount

## 1. Create the database and tables

In [3]:
server = {key: value for key, value in DB_CONFIG.items() if key != "database"}
conn = pymysql.connect(**server, charset="utf8mb4")
cur = conn.cursor()

cur.execute(f"CREATE DATABASE IF NOT EXISTS {DB_CONFIG['database']} "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
cur.execute(f"USE {DB_CONFIG['database']}")
for statement in SQL_FILE.read_text(encoding="utf-8").split(";"):
    # drop comment lines so a chunk that is only comments isn't executed
    code = "\n".join(line for line in statement.splitlines() if not line.strip().startswith("--"))
    if code.strip():
        cur.execute(code)
conn.commit()

cur.execute("SHOW TABLES")
print("Tables:", [row[0] for row in cur.fetchall()])

Tables: ['game_stats', 'games', 'goalie_season_stats', 'players', 'skater_season_stats', 'standings', 'teams']


## 2. Teams and standings

One call to `/standings/now`. The API has no numeric team id, so `teams` is inserted first and the ids MySQL generated are read back into `team_ids` (abbreviation -> `team_id`). Every other table uses that lookup.

In [4]:
standings = get_json(f"{BASE_URL}/standings/now")["standings"]

teams_df = make_frame([{
    "team_abbrev": name(row["teamAbbrev"]),
    "team_name": name(row["teamName"]),
    "conference_name": row.get("conferenceName"),
    "division_name": row.get("divisionName"),
    "logo_url": row.get("teamLogo"),
} for row in standings], key=["team_abbrev"])

print("teams inserted:", insert_rows(cur, "teams", teams_df))
cur.execute("SELECT team_abbrev, team_id FROM teams")
team_ids = dict(cur.fetchall())
teams = sorted(team_ids)

standings_df = make_frame([{
    "team_id": team_ids[name(row["teamAbbrev"])],
    "season": str(row["seasonId"]),
    "games_played": row.get("gamesPlayed"),
    "wins": row.get("wins"),
    "losses": row.get("losses"),
    "ot_losses": row.get("otLosses"),
    "points": row.get("points"),
    "goals_for": row.get("goalFor"),
    "goals_against": row.get("goalAgainst"),
    "home_wins": row.get("homeWins"),
    "away_wins": row.get("roadWins"),
    "streak_type": row.get("streakCode"),
    "streak_count": row.get("streakCount"),
} for row in standings], key=["team_id", "season"],
   ints=["games_played", "wins", "losses", "ot_losses", "points", "goals_for", "goals_against",
         "home_wins", "away_wins", "streak_count"])

print("standings inserted:", insert_rows(cur, "standings", standings_df))
conn.commit()
teams_df.head()

teams inserted: 32
standings inserted: 32


,team_abbrev,team_name,conference_name,division_name,logo_url
0,COL,Colorado Avalanche,Western,Central,https://assets.nhle.com/logos/nhl/svg/COL_ligh...
1,CAR,Carolina Hurricanes,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...
2,DAL,Dallas Stars,Western,Central,https://assets.nhle.com/logos/nhl/svg/DAL_ligh...
3,BUF,Buffalo Sabres,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BUF_ligh...
4,TBL,Tampa Bay Lightning,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/TBL_ligh...


## 3. Players from the current rosters

One call per team. The `forwards`, `defensemen` and `goalies` lists are flattened into one set of rows; `position` tells them apart. `L` / `R` become `LW` / `RW` to match the project spec.

In [5]:
POSITIONS = {"L": "LW", "R": "RW"}
PLAYER_TYPES = {"key": ["player_id"], "ints": ["player_id", "team_id", "jersey_number"],
                "floats": ["height_cm", "weight_kg"], "dates": ["birth_date"]}


def player_row(player, player_id, team_id, position):
    return {
        "player_id": player_id,
        "team_id": team_id,
        "first_name": name(player.get("firstName")),
        "last_name": name(player.get("lastName")),
        "position": POSITIONS.get(position, position),
        "jersey_number": player.get("sweaterNumber"),
        "birth_date": player.get("birthDate"),
        "birth_country": player.get("birthCountry"),
        "height_cm": player.get("heightInCentimeters"),
        "weight_kg": player.get("weightInKilograms"),
        "shoots_catches": player.get("shootsCatches"),
        "headshot_url": player.get("headshot"),
    }


def clean_players(rows):
    df = make_frame(rows, **PLAYER_TYPES)
    if df.empty:
        return df
    return df[df["first_name"].notna() & df["last_name"].notna()]


inserted = 0
for team in teams:
    roster = get_json(f"{BASE_URL}/roster/{team}/current")
    if roster is None:
        print("No roster for", team)
        continue
    rows = [player_row(p, p["id"], team_ids[team], p.get("positionCode"))
            for group in ("forwards", "defensemen", "goalies")
            for p in roster.get(group, [])]
    inserted += insert_rows(cur, "players", clean_players(rows))
conn.commit()

cur.execute("SELECT player_id FROM players")
known_players = {row[0] for row in cur.fetchall()}
print("players inserted:", inserted, "| players in table:", len(known_players))

players inserted: 979 | players in table: 979


### Players missing from the current rosters

`/roster/{team}/current` is the off-season roster, so players who were traded, released or retired after 2025-26 (and preseason call-ups) aren't in it, even though they have stats. Before any stats row is inserted, its players are checked against `known_players`; anyone missing is fetched from `/player/{id}/landing` (same bio fields) and added with the team they played for. Stats rows whose player still can't be found are dropped so the foreign key is never broken.

In [6]:
added_players = 0


def add_missing_players(player_ids, team_id):
    global added_players
    rows = []
    for player_id in dict.fromkeys(player_ids):
        if player_id in known_players:
            continue
        player = get_json(f"{BASE_URL}/player/{player_id}/landing")
        if player is not None:
            rows.append(player_row(player, player_id, team_id, player.get("position")))
    df = clean_players(rows)
    if not df.empty:
        added_players += insert_rows(cur, "players", df)
        known_players.update(int(pid) for pid in df["player_id"])

## 4. Games

One schedule call per team. Each game comes back from both teams' schedules, so roughly half of the rows are skipped by `INSERT IGNORE` - comparing "rows from the API" with "inserted" shows that. Games against a team that isn't one of the 32 NHL clubs can't satisfy the foreign key and are skipped.

In [7]:
rows_from_api, inserted, not_nhl = 0, 0, 0
for team in teams:
    schedule = get_json(f"{BASE_URL}/club-schedule-season/{team}/{SEASON}")
    if schedule is None:
        print("No schedule for", team)
        continue
    rows = []
    for game in schedule.get("games", []):
        rows_from_api += 1
        home, away = game["homeTeam"], game["awayTeam"]
        if home["abbrev"] not in team_ids or away["abbrev"] not in team_ids:
            not_nhl += 1
            continue
        rows.append({
            "game_id": game["id"],
            "season": str(game["season"]),
            "game_type": game["gameType"],
            "game_date": game["gameDate"],
            "home_team_id": team_ids[home["abbrev"]],
            "away_team_id": team_ids[away["abbrev"]],
            "home_score": home.get("score"),
            "away_score": away.get("score"),
            "game_state": game.get("gameState"),
            "venue_name": name(game.get("venue")),
        })
    df = make_frame(rows, key=["game_id"], dates=["game_date"],
                    ints=["game_id", "game_type", "home_team_id", "away_team_id", "home_score", "away_score"])
    if not df.empty:
        df = df[df["game_date"].notna()]
    inserted += insert_rows(cur, "games", df)
conn.commit()

print("game rows from the API:", rows_from_api)
print("skipped (opponent not an NHL team):", not_nhl)
print("games inserted:", inserted)

game rows from the API: 2996
skipped (opponent not an NHL team): 0
games inserted: 1498


## 5. Skater and goalie season stats

One club-stats call per team (regular season). The response has skaters and goalies together: position `G` goes to `goalie_season_stats`, every other position to `skater_season_stats`. Average time on ice comes in seconds and is converted to `MM:SS`.

In [8]:
skater_count, goalie_count = 0, 0
for team in teams:
    club = get_json(f"{BASE_URL}/club-stats/{team}/{SEASON}/2")
    if club is None:
        print("No club stats for", team)
        continue
    team_id = team_ids[team]
    season = str(club.get("season", SEASON))
    players = club.get("skaters", []) + club.get("goalies", [])
    add_missing_players([p["playerId"] for p in players], team_id)

    skater_rows, goalie_rows = [], []
    for p in players:
        if p["playerId"] not in known_players:
            continue
        base = {"player_id": p["playerId"], "season": season, "team_id": team_id}
        if p.get("positionCode", "G") == "G":
            goalie_rows.append({**base,
                "games_played": p.get("gamesPlayed"),
                "wins": p.get("wins"),
                "losses": p.get("losses"),
                "ot_losses": p.get("overtimeLosses"),
                "save_pct": p.get("savePercentage"),
                "goals_against_avg": p.get("goalsAgainstAverage"),
                "shutouts": p.get("shutouts"),
                "saves": p.get("saves"),
            })
        else:
            skater_rows.append({**base,
                "games_played": p.get("gamesPlayed"),
                "goals": p.get("goals"),
                "assists": p.get("assists"),
                "points": p.get("points"),
                "plus_minus": p.get("plusMinus"),
                "penalty_min": p.get("penaltyMinutes"),
                "shots": p.get("shots"),
                "avg_toi": toi_to_mmss(p.get("avgTimeOnIcePerGame")),
            })

    key = ["player_id", "season", "team_id"]
    skater_count += insert_rows(cur, "skater_season_stats", make_frame(
        skater_rows, key, ints=["player_id", "team_id", "games_played", "goals", "assists", "points",
                                "plus_minus", "penalty_min", "shots"]))
    goalie_count += insert_rows(cur, "goalie_season_stats", make_frame(
        goalie_rows, key, ints=["player_id", "team_id", "games_played", "wins", "losses", "ot_losses",
                                "shutouts", "saves"],
        floats=["save_pct", "goals_against_avg"]))
    conn.commit()

print("skater season rows inserted:", skater_count)
print("goalie season rows inserted:", goalie_count)
print("players added from the landing endpoint so far:", added_players)

skater season rows inserted: 1023
goalie season rows inserted: 100
players added from the landing endpoint so far: 190


## 6. Game stats from boxscores

One boxscore call for every finished game in `games`. Only the `forwards` and `defense` lists are used (the table is skaters only), and the team is whichever side the player is listed under.

This is the long part - well over a thousand requests with the delay in between (roughly 25-30 minutes). Rows are committed every 50 games, and games that already have stats in the table are skipped, so if the loop is interrupted it can simply be run again and it carries on where it stopped.

In [9]:
cur.execute("SELECT game_id FROM games WHERE game_state IN ('OFF', 'FINAL') ORDER BY game_id")
finished_games = [row[0] for row in cur.fetchall()]
cur.execute("SELECT DISTINCT game_id FROM game_stats")
already_loaded = {row[0] for row in cur.fetchall()}
game_ids = [gid for gid in finished_games if gid not in already_loaded]
print(f"Finished games: {len(finished_games)} | already loaded: {len(already_loaded)} | to fetch: {len(game_ids)}")

GAME_STAT_INTS = ["game_id", "player_id", "team_id", "goals", "assists", "points",
                  "shots_on_goal", "penalty_min", "plus_minus"]
inserted, failed = 0, []

for i, game_id in enumerate(game_ids, start=1):
    box = get_json(f"{BASE_URL}/gamecenter/{game_id}/boxscore")
    if box is None:
        failed.append(game_id)
        continue

    rows = []
    for side in ("homeTeam", "awayTeam"):
        team_id = team_ids.get(box[side]["abbrev"])
        groups = box.get("playerByGameStats", {}).get(side, {})
        skaters = groups.get("forwards", []) + groups.get("defense", [])
        if team_id is None or not skaters:
            continue
        add_missing_players([p["playerId"] for p in skaters], team_id)
        for p in skaters:
            if p["playerId"] not in known_players:
                continue
            rows.append({
                "game_id": game_id,
                "player_id": p["playerId"],
                "team_id": team_id,
                "goals": p.get("goals"),
                "assists": p.get("assists"),
                "points": p.get("points"),
                "shots_on_goal": p.get("sog"),
                "penalty_min": p.get("pim"),
                "toi": p.get("toi"),
                "plus_minus": p.get("plusMinus"),
            })

    inserted += insert_rows(cur, "game_stats", make_frame(rows, ["game_id", "player_id"], ints=GAME_STAT_INTS))
    if i % 50 == 0:
        conn.commit()
    if i % 200 == 0:
        print(f"{i}/{len(game_ids)} games done, {inserted} rows inserted")
conn.commit()

print("game_stats rows inserted:", inserted)
print("players added from the landing endpoint in total:", added_players)
print("boxscores that could not be fetched:", failed or "none")

Finished games: 1498 | already loaded: 0 | to fetch: 1498


200/1498 games done, 7200 rows inserted


400/1498 games done, 14400 rows inserted


600/1498 games done, 21600 rows inserted


800/1498 games done, 28800 rows inserted


1000/1498 games done, 36000 rows inserted


1200/1498 games done, 43199 rows inserted


1400/1498 games done, 50399 rows inserted


game_stats rows inserted: 53927
players added from the landing endpoint in total: 566
boxscores that could not be fetched: none


## 7. Checking the load

1. Row counts for every table (32 teams, 32 standings rows).
2. Orphan checks: a LEFT JOIN from each child table to its parent should find nothing missing.
3. Inserting the same API data again should add 0 rows, which shows `INSERT IGNORE` skipping duplicates.
4. A few values compared with a fresh call to the API.

In [10]:
TABLES = ["teams", "standings", "players", "games", "game_stats", "skater_season_stats", "goalie_season_stats"]


def table_counts():
    counts = {}
    for table in TABLES:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        counts[table] = cur.fetchone()[0]
    return counts


counts_after_load = table_counts()
pd.Series(counts_after_load, name="rows").to_frame()

,rows
teams,32
standings,32
players,1545
games,1498
game_stats,53927
skater_season_stats,1023
goalie_season_stats,100


In [11]:
orphan_checks = {
    "standings.team_id": "SELECT COUNT(*) FROM standings s LEFT JOIN teams t ON s.team_id = t.team_id WHERE t.team_id IS NULL",
    "players.team_id": "SELECT COUNT(*) FROM players p LEFT JOIN teams t ON p.team_id = t.team_id WHERE p.team_id IS NOT NULL AND t.team_id IS NULL",
    "games.home_team_id": "SELECT COUNT(*) FROM games g LEFT JOIN teams t ON g.home_team_id = t.team_id WHERE t.team_id IS NULL",
    "games.away_team_id": "SELECT COUNT(*) FROM games g LEFT JOIN teams t ON g.away_team_id = t.team_id WHERE t.team_id IS NULL",
    "game_stats.game_id": "SELECT COUNT(*) FROM game_stats gs LEFT JOIN games g ON gs.game_id = g.game_id WHERE g.game_id IS NULL",
    "game_stats.player_id": "SELECT COUNT(*) FROM game_stats gs LEFT JOIN players p ON gs.player_id = p.player_id WHERE p.player_id IS NULL",
    "skater_season_stats.player_id": "SELECT COUNT(*) FROM skater_season_stats ss LEFT JOIN players p ON ss.player_id = p.player_id WHERE p.player_id IS NULL",
    "goalie_season_stats.player_id": "SELECT COUNT(*) FROM goalie_season_stats gs LEFT JOIN players p ON gs.player_id = p.player_id WHERE p.player_id IS NULL",
}
for label, sql in orphan_checks.items():
    cur.execute(sql)
    print(f"{label:32} orphans: {cur.fetchone()[0]}")

standings.team_id                orphans: 0
players.team_id                  orphans: 0
games.home_team_id               orphans: 0
games.away_team_id               orphans: 0
game_stats.game_id               orphans: 0
game_stats.player_id             orphans: 0
skater_season_stats.player_id    orphans: 0
goalie_season_stats.player_id    orphans: 0


In [12]:
# insert the same data again: teams, standings, and one team's roster and schedule
print("teams inserted again:", insert_rows(cur, "teams", teams_df))
print("standings inserted again:", insert_rows(cur, "standings", standings_df))

roster = get_json(f"{BASE_URL}/roster/COL/current")
rows = [player_row(p, p["id"], team_ids["COL"], p.get("positionCode"))
        for group in ("forwards", "defensemen", "goalies") for p in roster.get(group, [])]
print("COL players inserted again:", insert_rows(cur, "players", clean_players(rows)))

schedule = get_json(f"{BASE_URL}/club-schedule-season/COL/{SEASON}")
rows = [{"game_id": g["id"], "season": str(g["season"]), "game_type": g["gameType"], "game_date": g["gameDate"],
         "home_team_id": team_ids[g["homeTeam"]["abbrev"]], "away_team_id": team_ids[g["awayTeam"]["abbrev"]],
         "home_score": g["homeTeam"].get("score"), "away_score": g["awayTeam"].get("score"),
         "game_state": g.get("gameState"), "venue_name": name(g.get("venue"))}
        for g in schedule["games"]
        if g["homeTeam"]["abbrev"] in team_ids and g["awayTeam"]["abbrev"] in team_ids]
print("COL games inserted again:", insert_rows(cur, "games", make_frame(rows, ["game_id"], dates=["game_date"])))
conn.commit()

print("Counts unchanged:", table_counts() == counts_after_load)

teams inserted again: 0
standings inserted again: 0


COL players inserted again: 0


COL games inserted again: 0
Counts unchanged: True


In [13]:
live = get_json(f"{BASE_URL}/standings/now")["standings"]
api_points = {name(row["teamAbbrev"]): row["points"] for row in live}
cur.execute("SELECT t.team_abbrev, s.points FROM standings s JOIN teams t ON s.team_id = t.team_id")
print("Team points match the API for all teams:", dict(cur.fetchall()) == api_points)

club = get_json(f"{BASE_URL}/club-stats/COL/{SEASON}/2")
api_top = max(club["skaters"], key=lambda p: p["points"])
cur.execute("""
    SELECT p.first_name, p.last_name, ss.points
    FROM skater_season_stats ss
    JOIN players p ON ss.player_id = p.player_id
    WHERE ss.team_id = %s
    ORDER BY ss.points DESC
    LIMIT 1
""", [team_ids["COL"]])
print("COL top scorer in the database:", cur.fetchone())
print("COL top scorer from the API:   ", (name(api_top["firstName"]), name(api_top["lastName"]), api_top["points"]))

Team points match the API for all teams: True


COL top scorer in the database: ('Nathan', 'MacKinnon', 127)
COL top scorer from the API:    ('Nathan', 'MacKinnon', 127)


In [14]:
cur.close()
conn.close()